# 1. Data Cleaning: Global Tech Salaries

This notebook prepares the salary dataset for exploratory analysis and modeling.

The dataset is global rather than specifically Tunisian. We will preserve the Tunisian records for a separate comparison, but we will not treat them as a representative Tunisian sample.

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("../ds_salaries.csv")
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

In [9]:
df_raw = pd.read_csv(DATA_PATH, index_col=0)

print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
display(df_raw.head())
df_raw.info()

Rows: 607
Columns: 11


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


<class 'pandas.DataFrame'>
RangeIndex: 607 entries, 0 to 606
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   work_year           607 non-null    int64
 1   experience_level    607 non-null    str  
 2   employment_type     607 non-null    str  
 3   job_title           607 non-null    str  
 4   salary              607 non-null    int64
 5   salary_currency     607 non-null    str  
 6   salary_in_usd       607 non-null    int64
 7   employee_residence  607 non-null    str  
 8   remote_ratio        607 non-null    int64
 9   company_location    607 non-null    str  
 10  company_size        607 non-null    str  
dtypes: int64(4), str(7)
memory usage: 52.3 KB


In [3]:
missing = df_raw.isna().sum().sort_values(ascending=False)
duplicates = df_raw.duplicated().sum()

print("Missing values by column:")
display(missing[missing > 0])
print(f"Duplicate rows: {duplicates}")

print("Year range:", df_raw["work_year"].min(), "to", df_raw["work_year"].max())
print("Salary values below or equal to zero:", (df_raw["salary_in_usd"] <= 0).sum())
print("Unique currencies:", df_raw["salary_currency"].nunique())

Missing values by column:


Series([], dtype: int64)

Duplicate rows: 42
Year range: 2020 to 2022
Salary values below or equal to zero: 0
Unique currencies: 17


## Cleaning decisions

- Remove exact duplicate rows.
- Remove rows missing essential fields such as job title, experience level, or USD salary.
- Keep `salary_in_usd` as the comparison column because original salaries use multiple currencies.
- Keep the original salary and currency columns for traceability.
- Add readable labels for experience, employment type, and remote-work status.

In [4]:
df = df_raw.copy()

df = df.drop_duplicates().copy()

required_columns = [
    "work_year",
    "experience_level",
    "employment_type",
    "job_title",
    "salary_in_usd",
    "employee_residence",
    "company_location",
    "remote_ratio",
    "company_size",
]
df = df.dropna(subset=required_columns).copy()
df = df[df["salary_in_usd"] > 0].copy()

print(f"Rows after cleaning: {len(df)}")
print(f"Rows removed: {len(df_raw) - len(df)}")

Rows after cleaning: 565
Rows removed: 42


In [10]:
experience_map = {
    "EN": "Entry-level",
    "MI": "Mid-level",
    "SE": "Senior-level",
    "EX": "Executive",
}

employment_map = {
    "FT": "Full-time",
    "PT": "Part-time",
    "CT": "Contract",
    "FL": "Freelance",
}

remote_map = {
    0: "On-site",
    50: "Hybrid",
    100: "Fully remote",
}

df["experience_name"] = df["experience_level"].map(experience_map).fillna("Other")
df["employment_name"] = df["employment_type"].map(employment_map).fillna("Other")
df["remote_name"] = df["remote_ratio"].map(remote_map).fillna("Unknown")
df["salary_log_usd"] = np.log1p(df["salary_in_usd"])

display(df[[
    "job_title",
    "experience_name",
    "employment_name",
    "remote_name",
    "salary_in_usd",
    "salary_log_usd",
    "company_location",
 ]].head())

,job_title,experience_name,employment_name,remote_name,salary_in_usd,salary_log_usd,company_location
0,Data Scientist,Mid-level,Full-time,On-site,79833,11.29,DE
1,Machine Learning Scientist,Senior-level,Full-time,On-site,260000,12.47,JP
2,Big Data Engineer,Senior-level,Full-time,Hybrid,109024,11.60,GB
3,Product Data Analyst,Mid-level,Full-time,On-site,20000,9.90,HN
4,Machine Learning Engineer,Senior-level,Full-time,Hybrid,150000,11.92,US


In [11]:
continent_map = {
    # Africa
    "DZ": "Africa", "KE": "Africa", "NG": "Africa", "TN": "Africa",
    # Asia
    "AE": "Asia", "CN": "Asia", "HK": "Asia", "IL": "Asia",
    "IN": "Asia", "IQ": "Asia", "IR": "Asia", "JP": "Asia",
    "MY": "Asia", "PK": "Asia", "SG": "Asia", "VN": "Asia",
    # Europe
    "AT": "Europe", "BE": "Europe", "CZ": "Europe", "DE": "Europe",
    "DK": "Europe", "EE": "Europe", "ES": "Europe", "FR": "Europe",
    "GB": "Europe", "GR": "Europe", "HR": "Europe", "IE": "Europe",
    "IT": "Europe", "LU": "Europe", "MT": "Europe", "NL": "Europe",
    "PL": "Europe", "PT": "Europe", "RO": "Europe", "RS": "Europe",
    "RU": "Europe", "SI": "Europe", "UA": "Europe",
    # North America
    "CA": "North America", "HN": "North America", "MX": "North America",
    "US": "North America",
    # South America
    "AR": "South America", "BO": "South America", "BR": "South America",
    "CL": "South America", "CO": "South America",
    # Oceania
    "AU": "Oceania", "NZ": "Oceania",
}

df["employee_continent"] = df["employee_residence"].map(continent_map).fillna("Other")
df["company_continent"] = df["company_location"].map(continent_map).fillna("Other")
df["tunisian_record"] = (
    (df["employee_residence"] == "TN")
    | (df["company_location"] == "TN")
 )

print("Jobs by company continent:")
display(df["company_continent"].value_counts().rename("job_count"))

print("Median salary by company continent:")
continent_salary = (
    df.groupby("company_continent")["salary_in_usd"]
    .agg(job_count="count", median_salary_usd="median")
    .sort_values("median_salary_usd", ascending=False)
 )
display(continent_salary)

print(f"Records involving Tunisia: {df['tunisian_record'].sum()}")
display(df[df["tunisian_record"]][[
    "job_title",
    "salary_in_usd",
    "employee_residence",
    "company_location",
    "employee_continent",
    "company_continent",
    "remote_name",
 ]])

Jobs by company continent:


company_continent
North America    350
Europe           150
Asia              44
Other              8
South America      5
Oceania            4
Africa             4
Name: job_count, dtype: int64

Median salary by company continent:


,job_count,median_salary_usd
company_continent,,
North America,350,130000.00
Oceania,4,106212.50
Europe,150,63831.00
Asia,44,31021.50
Africa,4,30000.00
South America,5,21844.00
Other,8,19112.00


Records involving Tunisia: 1


,job_title,salary_in_usd,employee_residence,company_location,employee_continent,company_continent,remote_name
489,Applied Machine Learning Scientist,31875,TN,CZ,Africa,Europe,Fully remote


In [12]:
output_path = OUTPUT_DIR / "ds_salaries_clean.csv"
df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print(f"Final rows: {df.shape[0]}")
print(f"Final columns: {df.shape[1]}")

Saved cleaned dataset to: ..\data\ds_salaries_clean.csv
Final rows: 565
Final columns: 18
